# ITA106 - LAB 7: Biểu đồ tương tác nâng cao với Plotly, Dash và Streamlit


## Bài 1: Hiển thị mối quan hệ giữa hàm lượng Alcohol và Malic Acid, phân biệt màu sắc theo loại rượu.


In [1]:
import pandas as pd
import plotly.express as px
from sklearn.datasets import load_wine

wine = load_wine()
df = pd.DataFrame(wine.data, columns=wine.feature_names)
df['target'] = wine.target.astype(str) # Chuyển sang chuỗi ký tự hiển thị dạng danh mục

fig = px.scatter(
    df, 
    x="alcohol", 
    y="malic_acid", 
    color="target", 
    title="Interactive Scatter Plot - Wine Dataset"
)
fig.show()


**Giải thích Code & Kết quả Bài 1:**
- **Code:** `px.scatter` liên kết thuộc tính trục tọa độ x là hàm lượng cồn (`alcohol`), trục y là lượng axit hữu cơ (`malic_acid`), tô màu dựa theo nhãn lớp phân loại rượu vang (`target`).
- **Kết quả:** Một đồ thị phân tán động. Người dùng có thể rê chuột vào từng điểm chấm tròn để xem chính xác thông số chi tiết (hover layout), phóng to khoanh vùng cụ thể hoặc chọn ẩn đi các nhóm phân loại trên bảng phân phối legend để dễ đối chiếu.


### Bài 1 Mở rộng: Nâng cấp trực quan hóa đa chiều 
**Cải tiến:**
1. Thêm chiều dữ liệu thứ 4: Kích thước các chấm tròn động dựa trên nồng độ khoáng chất của tro bụi (`size="ash"`).
2. Tích hợp đường xu hướng hồi quy tuyến tính (`trendline="ols"`) để nhận diện ngay xu hướng tương quan cục bộ của từng nhóm rượu vang.


In [2]:
fig_ext = px.scatter(
    df, 
    x="alcohol", 
    y="malic_acid", 
    color="target",
    size="ash", 
    hover_name="target",
    labels={"alcohol": "Nồng độ Cồn (%)", "malic_acid": "Hàm lượng Axit Malic", "target": "Phân loại Rượu"},
    title="Bài 1 Mở rộng: Phân tích Đa chiều Dữ liệu Rượu Vang (Alcohol vs Malic Acid vs Ash)",
    trendline="ols" # Thêm đường xu hướng OLS hồi quy tuyến tính
)
fig_ext.show()


**Giải thích Code & Kết quả Bài 1 mở rộng:**
- **Cải tiến:** Việc lồng ghép tham số `size="ash"` giúp mô tả thêm một chiều thuộc tính vật lý của dữ liệu mà không cần thêm biểu đồ phụ. Thuộc tính `trendline="ols"` tự động thực hiện phép toán bình phương tối thiểu để kiểm tra xu hướng đồng biến nghịch biến của nồng độ cồn và axit.
- **Kết quả:** Trực quan hóa trở nên giàu thông tin, giúp người phân tích nhận ra ngay lớp rượu vang số '0' có mật độ cồn cao vượt trội đồng thời mối quan hệ giữa cồn và axit dốc hơn các lớp còn lại.


## Bài 2: Linking Visualization với Dash Framework
**Yêu cầu:** Thiết lập tương tác liên kết động (Linking). Khi thực hiện khoanh vùng chọn dữ liệu (Lasso / Box Select) trên biểu đồ phân tán Scatter, biểu đồ cột Histogram kế bên tự động tính toán lại và chỉ hiển thị phân phối của vùng được chọn.


In [3]:
from dash import Dash, dcc, html, Input, Output
import plotly.express as px
import pandas as pd
from sklearn.datasets import load_wine

# 1. Khởi tạo dữ liệu
wine = load_wine()
df = pd.DataFrame(wine.data, columns=wine.feature_names)
df['target'] = wine.target.astype(str)

# Khởi tạo sẵn biểu đồ Scatter mặc định ban đầu
fig_scatter = px.scatter(df, x="alcohol", y="malic_acid", color="target")

# 2. Tạo app Dash hỗ trợ chạy inline trên Notebook
app = Dash(__name__)

# ĐƯA FIG_SCATTER VÀO THẲNG LAYOUT ĐỂ TRÁNH ĐÈ CHÈN LÊN TIÊU ĐỀ
app.layout = html.Div([
    html.H2(
        "Bài 2: Tương tác Liên kết Biểu đồ động (Linking)", 
        style={
            'textAlign': 'center', 
            'marginBottom': '20px', 
            'color': '#111111',
            'fontWeight': 'bold'
        }
    ),
    dcc.Graph(id="scatter", figure=fig_scatter),
    dcc.Graph(id="histogram")
])

# 3. Định nghĩa hàm xử lý tương tác (Callback)
@app.callback(
    Output("histogram", "figure"),
    Input("scatter", "selectedData")
)
def update_histogram(selectedData):
    if selectedData is None or not selectedData['points']:
        filtered_df = df
    else:
        # Lấy chỉ số thứ tự dòng dữ liệu (pointIndex) xuất hiện trong vùng khoanh vùng tương tác
        points = [p["pointIndex"] for p in selectedData["points"]]
        filtered_df = df.iloc[points]
        
    fig = px.histogram(
        filtered_df, 
        x="alcohol", 
        title="Phân phối tần suất Nồng độ Cồn của phân vùng được chọn"
    )
    return fig

if __name__ == '__main__':
    app.run(jupyter_mode='inline', port=8055)

**Giải thích Code & Kết quả Bài 2:**
- **Code:** Điểm mấu chốt là lắng nghe thay đổi của thuộc tính đầu vào `selectedData` từ thành phần `dcc.Graph(id="scatter")`. Danh sách chỉ số `pointIndex` được dùng làm bộ lọc chỉ mục dữ liệu thông qua câu lệnh `df.iloc[points]`.
- **Kết quả:** Biểu đồ Histogram thay đổi ngay lập tức theo thời gian thực (real-time) tương ứng với số lượng điểm mà bạn khoanh vùng trên biểu đồ Scatter Plot, giúp người phân tích cô lập dữ liệu nhiễu hiệu quả.


### Bài 2 Mở rộng: Đồng bộ hóa bố cục song song phối hợp đồ thị mật độ (Marginal Rug Plot)
**Cải tiến:**
1. Tái cấu trúc bố cục giao diện hiển thị sang dạng bảng cột song song để dễ quan sát đối chiếu không cần kéo cuộn chuột.
2. Thêm thanh mật độ tuyến tính `marginal="rug"` và đổ màu sắc phân tách nhóm trực tiếp trong Histogram giúp tối ưu mật độ thông tin biểu diễn.


In [4]:
app_ext = Dash(__name__)

# 1. Tạo sẵn biểu đồ Scatter chính ban đầu để truyền trực tiếp vào layout
fig_scatter_initial = px.scatter(df, x="alcohol", y="malic_acid", color="target", title="Quét khoanh vùng dữ liệu tại đây")

# 2. Định nghĩa bố cục Layout (Gán trực tiếp fig_scatter_initial vào tham số figure của dcc.Graph)
app_ext.layout = html.Table([
    html.Tr([
        html.Td(html.H2("Bài 2 Mở rộng: Liên kết Bố cục Song song Đa màu sắc", style={'fontFamily': 'Arial'}), colSpan=2)
    ]),
    html.Tr([
        # CẢI TIẾN: Truyền thẳng đối tượng đồ thị vào tham số figure ở đây
        html.Td(dcc.Graph(id="scatter_ext", figure=fig_scatter_initial), style={'width': '50%'}),
        html.Td(dcc.Graph(id="histogram_ext"), style={'width': '50%'})
    ])
], style={'width': '100%', 'borderCollapse': 'collapse'})

# 3. Hàm Callback xử lý logic tương tác liên kết (Giữ nguyên)
@app_ext.callback(
    Output("histogram_ext", "figure"),
    Input("scatter_ext", "selectedData")
)
def update_histogram_ext(selectedData):
    if selectedData is None or not selectedData['points']:
        filtered_df = df
    else:
        points = [p["pointIndex"] for p in selectedData["points"]]
        filtered_df = df.iloc[points]
    
    fig = px.histogram(
        filtered_df, 
        x="alcohol", 
        color="target",
        marginal="rug", 
        title="Mật độ phân bổ chi tiết Nồng độ Cồn",
        color_discrete_sequence=px.colors.qualitative.Dark24
    )
    return fig
if __name__ == '__main__':
    app_ext.run(jupyter_mode='inline', port=8060)

**Giải thích Code & Kết quả Bài 2 mở rộng:**
- **Cải tiến:** Sử dụng cấu trúc hiển thị bảng cố định giúp đồ thị nằm gọn gàng trên màn hình thiết bị. Thành phần `marginal="rug"` tạo ra các vạch nhỏ dưới đáy đồ thị, giúp người xem định vị chính xác vị trí chính xác của từng điểm lẻ, khắc phục nhược điểm gom nhóm mờ nhạt của histogram thông thường.
- **Kết quả:** Trải nghiệm liên kết phân tích mạch lạc, các nhóm màu sắc đồng nhất giúp theo dõi phân phối của từng loại rượu một cách tách biệt lập tức khi quét chọn vùng.


## Bài 3: Brushing and Filtering ứng dụng Dash Dropdown
**Yêu cầu:** Xây dựng tính năng lọc dữ liệu theo điều kiện danh mục lựa chọn (Dropdown). Người dùng lựa chọn xem riêng lẻ từng loại nhóm rượu vang hoặc xem tất cả.


In [5]:

wine = load_wine()
df = pd.DataFrame(wine.data, columns=wine.feature_names)
df['target'] = wine.target.astype(str)

fig_initial = px.scatter(df, x="alcohol", y="malic_acid", color="target", title="Biểu đồ phân tán tổng quan")

app_b3 = Dash(__name__)

# 2. Xây dựng bố cục Layout thoáng đãng, tiêu đề nổi bật không bị chìm
app_b3.layout = html.Div([
    html.H2(
        "Bài 3: Bộ lọc dữ liệu phân loại danh mục (Target Filter)",
        style={'textAlign': 'center', 'fontWeight': 'bold', 'color': '#111111', 'marginBottom': '15px'}
    ),
    
    html.Div([
        html.Label("Chọn loại rượu vang hiển thị:", style={'fontWeight': 'bold', 'marginBottom': '5px', 'display': 'block'}),
        dcc.Dropdown(
    id="target_filter",
    options=[
        {"label": "Xem tất cả các loại (All Varieties)", "value": "all"},
        {"label": "Rượu vang Thượng hạng (Class 0)", "value": "0"},
        {"label": "Rượu vang Phổ thông (Class 1)", "value": "1"},
        {"label": "Rượu vang Đậm vị Axit (Class 2)", "value": "2"}
    ],
    value="all",
    clearable=False
),
    ], style={'width': '50%', 'margin': '0 auto 25px auto'}), # Đưa thanh menu thả xuống ra giữa cho cân đối
    
    dcc.Graph(id="scatter_filtered", figure=fig_initial)
], style={'padding': '20px'})

# 3. Hàm Callback xử lý tương tác bộ lọc dữ liệu khi tương tác với Dropdown
@app_b3.callback(
    Output("scatter_filtered", "figure"),
    Input("target_filter", "value")
)
def update_chart(selected_target):
    if selected_target == "all":
        filtered_df = df
    else:
        filtered_df = df[df["target"] == selected_target]
        
    fig = px.scatter(
        filtered_df, 
        x="alcohol", 
        y="malic_acid", 
        color="target",
        title=f"Biểu đồ phân tán - Đang hiển thị trạng thái lọc: Loại {selected_target.upper()}"
    )
    return fig

# 4. CHẠY TRỰC TIẾP TRÊN JUPYTER NOTEBOOK (Đổi sang port 8070 an toàn)
if __name__ == '__main__':
    app_b3.run(jupyter_mode='inline', port=8070)

**Giải thích Code & Kết quả Bài 3:**
- **Code:** Nhận giá trị string từ Dropdown truyền trực tiếp vào bộ lọc dữ liệu logic của Pandas `df[df["target"] == selected_target]` để tái tạo dữ liệu cho hàm `px.scatter`.
- **Kết quả:** Toàn bộ điểm dữ liệu của các nhóm khác sẽ biến mất khỏi đồ thị, chỉ chừa lại nhóm được chỉ định, giúp người dùng tập trung so sánh hình dáng phân bố nội bộ của riêng lớp đó.


### Bài 3 Mở rộng: Bộ lọc Phối hợp Đa tầng nâng cao (Multi-Dropdown kết hợp RangeSlider số)
**Cải tiến:**
1. Đổi sang Dropdown đa chọn (`multi=True`) giúp hiển thị cùng lúc nhiều nhóm tùy chọn linh hoạt.
2. Thiết lập thêm thanh kéo dải số `RangeSlider` để lọc điều kiện giá trị liên tục nồng độ cồn (`alcohol`), đồng thời cố định khung tọa độ `range_x` của biểu đồ tránh cảm giác giật hình khi dữ liệu thay đổi.


In [6]:
wine = load_wine()
df = pd.DataFrame(wine.data, columns=wine.feature_names)
df['target'] = wine.target.astype(str)

min_alc = float(df["alcohol"].min())
max_alc = float(df["alcohol"].max())

# Khởi tạo sẵn biểu đồ nền ban đầu (Hiển thị tất cả khi mới load trang)
fig_initial = px.scatter(
    df, x="alcohol", y="malic_acid", color="target",
    title="Biểu đồ phân tán tổng quan chưa qua bộ lọc"
)

app_b3_ext = Dash(__name__)

# 2. Xây dựng bố cục Layout thoáng đãng, phân cấp rõ ràng
app_b3_ext.layout = html.Div([
    html.H2(
        "Bài 3 Mở rộng: Bộ lọc Phối hợp Liên hoàn Đa tầng Dữ liệu",
        style={'textAlign': 'center', 'fontWeight': 'bold', 'color': '#111111', 'marginBottom': '20px'}
    ),
    
    html.Div([
        html.Label("1. Lọc theo nhiều danh mục lớp rượu cùng lúc (Chọn được nhiều loại):", style={'fontWeight': 'bold', 'marginBottom': '5px', 'display': 'block'}),
        dcc.Dropdown(
            id="target_multi_filter",
            # CẬP NHẬT: Đặt tên tường minh và rõ ràng cho từng nhóm Class
            options=[
                {"label": "Rượu vang Thượng hạng (Class 0)", "value": "0"},
                {"label": "Rượu vang Phổ thông (Class 1)", "value": "1"},
                {"label": "Rượu vang Đậm vị Axit (Class 2)", "value": "2"}
            ],
            value=["0", "1", "2"], # Thiết lập mặc định ban đầu chọn tất cả
            multi=True
        ),
    ], style={'marginBottom': '25px'}),
    
    html.Div([
        html.Label("2. Lọc theo dải Nồng độ Cồn thực tế (Alcohol Continuous Range):", style={'fontWeight': 'bold', 'marginBottom': '5px', 'display': 'block'}),
        dcc.RangeSlider(
            id="alcohol_slider",
            min=min_alc,
            max=max_alc,
            step=0.1,
            value=[min_alc, max_alc],
            marks={int(i): f"{int(i)}%" for i in range(int(min_alc), int(max_alc)+1)} # Thêm ký tự % vào nhãn thanh trượt cho rõ nghĩa
        )
    ], style={'marginBottom': '25px'}),
    
    dcc.Graph(id="complex_scatter", figure=fig_initial)
], style={'padding': '25px'})

# 3. Hàm Callback xử lý logic kết hợp đa tầng dữ liệu
@app_b3_ext.callback(
    Output("complex_scatter", "figure"),
    [Input("target_multi_filter", "value"),
     Input("alcohol_slider", "value")]
)
def update_complex_chart(selected_targets, alcohol_range):
    # Lọc danh mục bằng phương thức .isin() của mảng pandas dataframe
    filtered_df = df[df["target"].isin(selected_targets)]
    
    # Lọc chuỗi liên tục bằng phép so sánh chặn trên và chặn dưới logic
    filtered_df = filtered_df[
        (filtered_df["alcohol"] >= alcohol_range[0]) & 
        (filtered_df["alcohol"] <= alcohol_range[1])
    ]
    
    fig = px.scatter(
        filtered_df, x="alcohol", y="malic_acid", color="target",
        title="Biểu đồ phân tán sau bộ lọc phối hợp đa thông số",
        range_x=[min_alc - 0.5, max_alc + 0.5] # Giữ nguyên chiều rộng trục X cố định giúp mắt điều tiết ổn định
    )
    return fig

if __name__ == '__main__':
    app_b3_ext.run(jupyter_mode='inline', port=8080)

**Giải thích Code & Kết quả Bài 3 mở rộng:**
- **Cải tiến:** Hàm callback nhận đồng thời một danh sách mảng đầu vào từ dropdown mục tiêu và một mảng 2 phần tử chứa cận min-max của slider. Phương thức lọc kết hợp chuỗi logic đa biến `.isin()` và toán tử bitwise `&`.
- **Kết quả:** Mang lại khả năng truy vấn dữ liệu sâu. Người dùng có thể dễ dàng giải quyết bài toán: *"Hãy so sánh hình ảnh phân bổ hóa học giữa loại rượu 0 và loại 2 trong phân khúc nồng độ cồn nhẹ từ 12% đến 13.5%."*


## Bài 4: Dashboard tương tác cho dữ liệu AI sử dụng Streamlit
**Yêu cầu:** Thiết kế ứng dụng Dashboard hoàn chỉnh bằng thư viện Streamlit hiển thị dữ liệu kết hợp bảng biểu phân tích tương tác theo bộ lọc đầu vào.


In [7]:
# !cd c:\Users\LENOVO\Downloads && python -m streamlit run app_streamlit.py
streamlit_code = '''
import streamlit as st
import pandas as pd
import plotly.express as px
from sklearn.datasets import load_wine

# 1. Cấu hình giao diện ứng dụng rộng toàn màn hình
st.set_page_config(page_title="Wine AI Dashboard", layout="wide")
st.title("Bài 4: Dashboard Phân tích Dữ liệu Wine với Streamlit (Cơ bản)")

# 2. Tải dữ liệu
# Tải data công khai và gộp cột chuẩn xác
wine = load_wine()
X = pd.DataFrame(wine.data, columns=wine.feature_names)
y = wine.target

# Tạo df tổng dùng cho vẽ đồ thị bên dưới
df = X.copy()
df['target'] = y

# 3. Tạo thanh Sidebar bên trái giao diện phục vụ lọc dữ liệu
st.sidebar.header("Thanh điều hướng bộ lọc")
species_choice = st.sidebar.selectbox("Chọn nhóm phân loại rượu:", ["Tất cả", 0, 1, 2])

if species_choice == "Tất cả":
    filtered_df = df
else:
    filtered_df = df[df["target"] == int(species_choice)]

# Tạo bản sao chuyển đổi mục tiêu sang định dạng chuỗi phục vụ trực quan hóa đẹp mắt
filtered_df_plot = filtered_df.copy()
filtered_df_plot['target'] = filtered_df_plot['target'].astype(str)

# 4. Sắp xếp bố cục hiển thị đồ thị dạng 2 cột song song
col1, col2 = st.columns(2)

with col1:
    st.subheader("Mối tương quan đặc trưng (Scatter Plot)")
    fig1 = px.scatter(filtered_df_plot, x="alcohol", y="malic_acid", color="target", title="Không gian Alcohol vs Malic Acid")
    st.plotly_chart(fig1, use_container_width=True)

with col2:
    st.subheader("Phân phối đặc trưng (Histogram Plot)")
    fig2 = px.histogram(filtered_df_plot, x="alcohol", title="Biểu đồ phân phối tần suất Cồn")
    st.plotly_chart(fig2, use_container_width=True)

# 5. Hiển thị bảng tra cứu dữ liệu gốc phía dưới
st.subheader("Bảng tra cứu chi tiết tập dữ liệu tương tác")
st.dataframe(filtered_df)
'''

# Ghi trực tiếp mã nguồn ra tệp tin hệ thống
with open("app_streamlit.py", "w", encoding="utf-8") as f:
    f.write(streamlit_code)
print(" thành công!")


 thành công!


In [8]:
import os
print(os.path.abspath("app_streamlit.py"))

e:\Visualization\Visualization\Visualization\Lab_main\Code\app_streamlit.py


In [9]:
#c
#python -m streamlit run app_streamlit.py

In [10]:
!cd c:\Users\LENOVO\Downloads && python -m streamlit run app_streamlit.py

^C


**Giải thích Code & Kết quả Bài 4:**
- **Code:** Dùng hàm `st.sidebar.selectbox` thu thập lựa chọn của người dùng. Layout chia cột thông qua `st.columns(2)` và dùng câu lệnh quản lý ngữ cảnh `with col1` để nhúng các biểu đồ plotly thông qua hàm `st.plotly_chart`.
- **Kết quả:** Tạo ra trang web phân tích trực quan nhanh chóng. Người dùng có thể lọc nhanh, dữ liệu hiển thị tức thì kết hợp bảng số liệu thô có khả năng tìm kiếm và sắp xếp trực tiếp rất thân thiện.
